# `run_training()` Integration Test
Tests the full training pipeline via the `run_training()` function:
```
s3://offline-store/parquet/users/staged.parquet
    ↓  DuckDB → entity_df (user_id + event_timestamp)
    ↓  Feast get_historical_features → feature_df
    ↓  X / y split  →  train_test_split
    ↓  model.fit()  →  MLflow log + register
    ↓  returns { run_id, model_version, status }
```
**Prerequisites:** Docker stack running + `preprocess.py` + `feast apply` + `run_materialization()` executed.

In [1]:
import os, sys

# Change to project root so relative paths (feast/, src/) resolve correctly
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# Set env vars BEFORE importing src — pydantic-settings reads them at instantiation time
os.environ['MLFLOW_TRACKING_URI']  = 'http://localhost:5001'   # Docker maps 5001→5000
os.environ.setdefault('AWS_ACCESS_KEY_ID',     'minioadmin')
os.environ.setdefault('AWS_SECRET_ACCESS_KEY', 'minioadmin123')
os.environ.setdefault('AWS_ENDPOINT_URL',      'http://localhost:9000')
os.environ.setdefault('AWS_S3_ENDPOINT',       'localhost:9000')

EXPERIMENT_NAME = 'test_run_training'
MODEL_TYPE      = 'random_forest'
FEATURE_VIEW    = 'raw_event_features'
# Keep estimators low so training on 770k rows finishes quickly
HYPERPARAMS     = {'n_estimators': 5, 'max_depth': 4, 'random_state': 42, 'n_jobs': -1}

print(f'Working directory : {os.getcwd()}')
print('Setup complete.')

Working directory : /Volumes/ktran2/MSE/MLops/MLOpsLabs
Setup complete.


## Test 1 — `run_training()` returns expected keys

In [2]:
from src.models.trainer import run_training

result = run_training(
    experiment_name=EXPERIMENT_NAME,
    model_type=MODEL_TYPE,
    hyperparams=HYPERPARAMS,
    feature_view=FEATURE_VIEW,
)

assert isinstance(result, dict),              'FAIL: result is not a dict'
assert 'run_id'        in result,             'FAIL: missing key run_id'
assert 'model_version' in result,             'FAIL: missing key model_version'
assert 'status'        in result,             'FAIL: missing key status'
assert result['status'] == 'registered',      f'FAIL: status={result["status"]!r}, expected "registered"'
assert isinstance(result['run_id'], str),     'FAIL: run_id is not a string'
assert isinstance(result['model_version'], int), 'FAIL: model_version is not an int'
assert result['model_version'] >= 1,          f'FAIL: model_version={result["model_version"]} < 1'

RUN_ID        = result['run_id']
MODEL_VERSION = result['model_version']

print('PASS: run_training() returned expected structure')
print(f'  run_id        : {RUN_ID}')
print(f'  model_version : {MODEL_VERSION}')
print(f'  status        : {result["status"]}')

/Volumes/ktran2/MSE/MLops/MLOpsLabs/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/04/18 21:30:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/18 21:30:07 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Volumes/ktran2/MSE/MLops/MLOpsLabs


2026/04/18 21:30:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/04/18 21:30:07 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Volumes/ktran2/MSE/MLops/MLOpsLabs


...

2026/04/18 21:30:07 WARNING mlflow.utils.uv_utils: uv is not available or version is below minimum required. Falling back to pip-based inference.


2026/04/18 21:30:07 WARNING mlflow.utils.environment: uv export failed or returned no requirements. Falling back to package capture based inference.


Registered model 'random_forest' already exists. Creating a new version of this model...
2026/04/18 21:30:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: random_forest, version 3


🏃 View run smiling-roo-910 at: http://localhost:5001/#/experiments/2/runs/f7211c2d051045a89dfe9d90bde457ff
🧪 View experiment at: http://localhost:5001/#/experiments/2
PASS: run_training() returned expected structure
  run_id        : f7211c2d051045a89dfe9d90bde457ff
  model_version : 3
  status        : registered


Created version '3' of model 'random_forest'.
/Volumes/ktran2/MSE/MLops/MLOpsLabs/src/models/trainer.py:71: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  versions = client.get_latest_versions(model_type)


## Test 2 — MLflow run exists with correct experiment and metrics

In [3]:
import mlflow
from src.core.config import settings

mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
client = mlflow.MlflowClient()

run = client.get_run(RUN_ID)

assert run is not None,                           'FAIL: MLflow run not found'
assert run.info.status == 'FINISHED',             f'FAIL: run status={run.info.status!r}'

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
assert experiment is not None,                    f'FAIL: experiment {EXPERIMENT_NAME!r} not found'
assert run.info.experiment_id == experiment.experiment_id, 'FAIL: run belongs to wrong experiment'

assert 'accuracy' in run.data.metrics,            'FAIL: accuracy metric not logged'
accuracy = run.data.metrics['accuracy']
assert 0.0 <= accuracy <= 1.0,                    f'FAIL: accuracy={accuracy} out of [0,1]'

assert run.data.params.get('model_type') == MODEL_TYPE, \
    f'FAIL: logged model_type={run.data.params.get("model_type")!r}'

print('PASS: MLflow run verified')
print(f'  Experiment : {EXPERIMENT_NAME}')
print(f'  Run status : {run.info.status}')
print(f'  Accuracy   : {accuracy:.4f}')
print(f'  Params     : {run.data.params}')

PASS: MLflow run verified
  Experiment : test_run_training
  Run status : FINISHED
  Accuracy   : 0.0037
  Params     : {'model_type': 'random_forest', 'feature_view': 'raw_event_features', 'n_estimators': '5', 'max_depth': '4', 'random_state': '42', 'n_jobs': '-1'}


## Test 3 — Model registered in MLflow registry

In [4]:
versions = client.get_latest_versions(MODEL_TYPE)

assert len(versions) > 0,  f'FAIL: no versions found for model {MODEL_TYPE!r}'

latest = max(versions, key=lambda v: int(v.version))
assert int(latest.version) == MODEL_VERSION, \
    f'FAIL: latest version={latest.version}, expected {MODEL_VERSION}'

print('PASS: Model registered in MLflow registry')
print(f'  Registered name : {MODEL_TYPE}')
print(f'  Latest version  : {latest.version}')
print(f'  Source run_id   : {latest.run_id}')

/var/folders/zb/w56hcx851rs25t5j8std72280000gn/T/ipykernel_32682/1522578955.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  versions = client.get_latest_versions(MODEL_TYPE)


PASS: Model registered in MLflow registry
  Registered name : random_forest
  Latest version  : 3
  Source run_id   : f7211c2d051045a89dfe9d90bde457ff


## Test 4 — Registered model loads and predicts

In [5]:
import pandas as pd

model_uri = f'models:/{MODEL_TYPE}/{MODEL_VERSION}'
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Probe with one synthetic row matching the training feature schema
# Features: gender_idx, age_idx, occupation, target_time  (target is the label, excluded from X)
sample_input = pd.DataFrame([{
    'gender_idx':  0,
    'age_idx':     1,
    'occupation':  4,
    'target_time': 1,
}])

preds = loaded_model.predict(sample_input)

assert preds is not None,     'FAIL: predict() returned None'
assert len(preds) == 1,       f'FAIL: expected 1 prediction, got {len(preds)}'

print('PASS: Registered model loads and predicts')
print(f'  Model URI  : {model_uri}')
print(f'  Sample input : {sample_input.to_dict("records")[0]}')
print(f'  Prediction   : {preds[0]}')

PASS: Registered model loads and predicts
  Model URI  : models:/random_forest/3
  Sample input : {'gender_idx': 0, 'age_idx': 1, 'occupation': 4, 'target_time': 1}
  Prediction   : 2599


## Summary

In [6]:
print('=' * 55)
print('  run_training() — ALL TESTS PASSED')
print('=' * 55)
print(f'  Experiment    : {EXPERIMENT_NAME}')
print(f'  Model type    : {MODEL_TYPE}')
print(f'  run_id        : {RUN_ID}')
print(f'  model_version : {MODEL_VERSION}')
print(f'  accuracy      : {accuracy:.4f}')
print('=' * 55)

  run_training() — ALL TESTS PASSED
  Experiment    : test_run_training
  Model type    : random_forest
  run_id        : f7211c2d051045a89dfe9d90bde457ff
  model_version : 3
  accuracy      : 0.0037
